# Smart Parking — kiểm thử tải trên Kaggle
Bật Internet, chọn CPU rồi Run All. Backend, MariaDB và Locust chạy cùng phiên Kaggle tại localhost.
Để test bản code trên máy: build JAR bằng `mvnw.cmd -DskipTests package`, upload JAR thành Kaggle Dataset và Add Input. Notebook ưu tiên JAR này; nếu không có sẽ build nhánh main công khai.
Kết quả đo phản ánh tài nguyên dùng chung của phiên Kaggle. MariaDB là cơ sở dữ liệu thay thế MySQL; chưa xác nhận tương thích trên Kaggle.


In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq openjdk-17-jdk-headless mariadb-server > /dev/null
pip install -q locust
java -version


In [ ]:
%%bash
set -e
service mariadb start || (mysqld_safe > /tmp/mysqld.log 2>&1 &) ; sleep 5
mysql -uroot <<'SQL'
CREATE DATABASE IF NOT EXISTS parking CHARACTER SET utf8mb4;
CREATE USER IF NOT EXISTS 'parking'@'localhost' IDENTIFIED BY 'loadtest-pass';
GRANT ALL ON parking.* TO 'parking'@'localhost';
FLUSH PRIVILEGES;
SQL


In [ ]:
from pathlib import Path
import subprocess
root=Path('/kaggle/working/app')
root.mkdir(exist_ok=True)
jars=list(Path('/kaggle/input').rglob('parking-*.jar'))
if jars:
    jar=str(jars[0])
    print('Using uploaded JAR:', jar)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/xuantung159/smartParking.git',str(root)],check=True)
    subprocess.run(['bash','mvnw','-q','-DskipTests','package'],cwd=root,check=True)
    jar=str(next((root/'target').glob('parking-*.jar')))

(root / "scripts").mkdir(exist_ok=True)
(root / "scripts/locustfile.py").write_text('"""Kịch bản kiểm thử tải bằng Locust (chỉ đọc, không làm bẩn dữ liệu).\n\nChạy: locust -f scripts/locustfile.py --headless -u 50 -r 5 -t 3m \\\n      --host http://127.0.0.1:8080 --csv /kaggle/working/load --html /kaggle/working/load.html\n"""\nfrom datetime import datetime, timedelta\nimport random\n\nfrom locust import HttpUser, between, task\n\n\ndef window():\n    start = datetime.now() + timedelta(days=random.randint(1, 7), hours=random.randint(0, 23))\n    end = start + timedelta(hours=random.randint(1, 4))\n    return dict(vehicleType=random.choice([\'CAR\', \'MOTORBIKE\']),\n                startTime=start.isoformat(timespec=\'seconds\'),\n                endTime=end.isoformat(timespec=\'seconds\'))\n\n\nclass Guest(HttpUser):\n    """Khách chưa đăng nhập: chỉ tra cứu chỗ trống (API công khai)."""\n    weight = 3\n    wait_time = between(0.5, 2)\n\n    @task\n    def availability(self):\n        self.client.get(\'/api/availability\', params=window(), name=\'/api/availability\')\n\n\nclass Staff(HttpUser):\n    """Nhân viên: đăng nhập một lần rồi xem phiên gửi xe, tra chỗ trống."""\n    weight = 1\n    wait_time = between(1, 3)\n\n    def on_start(self):\n        r = self.client.post(\'/api/auth/login\', json={\'username\': \'staff\', \'password\': \'123456\'})\n        r.raise_for_status()\n        self.client.headers[\'Authorization\'] = \'Bearer \' + r.json()[\'accessToken\']\n\n    @task(3)\n    def sessions(self):\n        self.client.get(\'/api/sessions\', params={\'status\': random.choice([\'ACTIVE\', \'COMPLETED\'])},\n                        name=\'/api/sessions?status\')\n\n    @task(2)\n    def availability(self):\n        self.client.get(\'/api/availability\', params=window(), name=\'/api/availability\')\n\n\nclass LoginStorm(HttpUser):\n    """Đo riêng chi phí đăng nhập (BCrypt tốn CPU). Bật bằng: locust ... LoginStorm"""\n    weight = 0\n    wait_time = between(1, 2)\n\n    @task\n    def login(self):\n        self.client.post(\'/api/auth/login\', json={\'username\': \'customer\', \'password\': \'123456\'})\n', encoding="utf-8")


In [ ]:
import subprocess, time, urllib.request, os

env = dict(os.environ,
           DB_URL='jdbc:mysql://localhost:3306/parking?allowPublicKeyRetrieval=true&useSSL=false',
           DB_USERNAME='parking', DB_PASSWORD='loadtest-pass',
           JWT_SECRET='kaggle-load-test-secret-at-least-32-chars', SERVER_PORT='8080')
app = subprocess.Popen(['java', '-Duser.timezone=Asia/Ho_Chi_Minh', '-Xms512m', '-Xmx2g', '-jar', jar],
                       env=env, stdout=open('/kaggle/working/app.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(90):                       # đợi tối đa ~90 s
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8080/actuator/health').read().decode()); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open('/kaggle/working/app.log').read()[-5000:])   # in log nếu không lên


In [ ]:
%%bash
set -e
cd /kaggle/working/app
for U in 10 50 100 200; do
  locust -f scripts/locustfile.py Guest Staff --headless \
         -u $U -r $((U/10>0?U/10:1)) -t 2m --host http://127.0.0.1:8080 \
         --csv /kaggle/working/load_u$U --html /kaggle/working/load_u$U.html --only-summary
done


In [ ]:
import pandas as pd, glob, re
rows = []
for f in sorted(glob.glob('/kaggle/working/load_u*_stats.csv'), key=lambda p: int(re.search(r'u(\d+)', p)[1])):
    df = pd.read_csv(f); agg = df[df['Name'] == 'Aggregated'].iloc[0]
    rows.append(dict(users=int(re.search(r'u(\d+)', f)[1]), rps=round(agg['Requests/s'], 1),
                     p50_ms=agg['50%'], p95_ms=agg['95%'], p99_ms=agg['99%'],
                     fail_pct=round(100 * agg['Failure Count'] / max(agg['Request Count'], 1), 2)))
summary = pd.DataFrame(rows)
summary.to_csv('/kaggle/working/load_summary.csv', index=False)
display(summary)


In [ ]:
summary.plot(x='users', y=['p50_ms', 'p95_ms'], marker='o', title='Độ trễ theo số user')
summary.plot(x='users', y='rps', marker='o', title='Thông lượng (req/s)')


In [ ]:
app.terminate()
